# MATH-Only Solver LoRA SFT

This notebook trains a fresh LoRA adapter for 3 epochs on the 2,000-example MATH full-trajectory dataset at `normalized_outputs/solver_full_trajectory_dataset.jsonl`, using `Qwen/Qwen2.5-0.5B-Instruct` as the base model.

Expected Google Drive layout:

```text
MyDrive/Final Project/
  newest_solver/
  normalized_outputs/solver_full_trajectory_dataset.jsonl
```


In [2]:
from google.colab import drive
from pathlib import Path
import os

drive.mount('/content/drive', force_remount=True)

PROJECT_ROOT = Path('/content/drive/MyDrive/Final Project')
SOLVER_DIR = PROJECT_ROOT / 'newest_solver'
NORMALIZED_DIR = PROJECT_ROOT / 'normalized_outputs'

if not SOLVER_DIR.exists():
    raise FileNotFoundError(f'Missing solver folder: {SOLVER_DIR}')
if not NORMALIZED_DIR.exists():
    raise FileNotFoundError(f'Missing normalized output folder: {NORMALIZED_DIR}')

os.chdir(SOLVER_DIR)
print('Working directory:', Path.cwd())
print('Normalized directory:', NORMALIZED_DIR)


Mounted at /content/drive
Working directory: /content/drive/MyDrive/Final Project/newest_solver
Normalized directory: /content/drive/MyDrive/Final Project/normalized_outputs


## Install Dependencies

In [9]:
!pip install -q -r requirements.txt rouge-score nltk pandas matplotlib "torchao>=0.16.0"

  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.2/3.2 MB 27.5 MB/s eta 0:00:00


## Prepare MATH-Only Data

In [10]:
SOURCE_DATASET = NORMALIZED_DIR / 'solver_full_trajectory_dataset.jsonl'

if not SOURCE_DATASET.exists():
    raise FileNotFoundError(f'Missing source normalized dataset: {SOURCE_DATASET}')

!python prepare_math_only_sft.py   --input "{SOURCE_DATASET}"   --output-dir data   --limit 2000   --skip-normalized-output

print('MATH full-trajectory source dataset:', SOURCE_DATASET)
!wc -l "{SOURCE_DATASET}" data/train_sft.jsonl data/val_sft.jsonl data/test_sft.jsonl


Skipped writing a filtered normalized JSONL copy.
normalized_math: 2000 examples, sources={'math': 2000}
train: 1800 examples, sources={'math': 1800}
val: 100 examples, sources={'math': 100}
test: 100 examples, sources={'math': 100}
Wrote SFT splits to /content/drive/MyDrive/Final Project/newest_solver/data
MATH full-trajectory source dataset: /content/drive/MyDrive/Final Project/normalized_outputs/solver_full_trajectory_dataset.jsonl
   2000 /content/drive/MyDrive/Final Project/normalized_outputs/solver_full_trajectory_dataset.jsonl
   1800 data/train_sft.jsonl
    100 data/val_sft.jsonl
    100 data/test_sft.jsonl
   4000 total


## Train LoRA For 3 Epochs

In [ ]:
# This trains from the original base model and saves the adapter to outputs/qwen2.5-0.5b-math-only-lora.
!python train_lora_sft.py --model Qwen/Qwen2.5-0.5B-Instruct --train-file data/train_sft.jsonl --val-file data/val_sft.jsonl --output-dir outputs/qwen2.5-0.5b-math-only-lora --max-seq-length 2048 --epochs 3 --per-device-train-batch-size 1 --per-device-eval-batch-size 1 --gradient-accumulation-steps 16 --learning-rate 2e-4 --lora-r 16 --lora-alpha 32 --lora-dropout 0.05 --fp16


Skipping import of cpp extensions due to incompatible torch version. Please upgrade to torch >= 2.11.0 (found 2.10.0+cu128).
config.json: 100% 659/659 [00:00<00:00, 3.25MB/s]
tokenizer_config.json: 7.30kB [00:00, 19.3MB/s]
vocab.json: 2.78MB [00:00, 73.3MB/s]
merges.txt: 1.67MB [00:00, 124MB/s]
tokenizer.json: 7.03MB [00:00, 143MB/s]
data/train_sft.jsonl: loaded 1800 encoded examples
data/val_sft.jsonl: loaded 100 encoded examples
`torch_dtype` is deprecated! Use `dtype` instead!
model.safetensors: 100% 988M/988M [00:08<00:00, 122MB/s]
Loading weights: 100% 290/290 [00:00<00:00, 298.56it/s, Materializing param=model.norm.weight]
generation_config.json: 100% 242/242 [00:00<00:00, 1.52MB/s]
trainable params: 8,798,208 || all params: 502,830,976 || trainable%: 1.7497
warmup_ratio is deprecated and will be removed in v5.2. Use `warmup_steps` instead.
{'loss': '0.7611', 'grad_norm': '0.904', 'learning_rate': '0.0001636', 'epoch': '0.08889'}
{'loss': '0.7081', 'grad_norm': '0.6357', 'learnin

## Load Base Model And MATH-Only Adapter

In [11]:
import json
import random
import torch
from pathlib import Path
from transformers import AutoModelForCausalLM, AutoTokenizer
from peft import PeftModel

BASE_MODEL = 'Qwen/Qwen2.5-0.5B-Instruct'
ADAPTER_DIR = Path('outputs/qwen2.5-0.5b-math-only-lora')
TEST_FILE = Path('data/test_sft.jsonl')

if not ADAPTER_DIR.exists():
    raise FileNotFoundError(f'Missing adapter directory: {ADAPTER_DIR.resolve()}')
if not TEST_FILE.exists():
    raise FileNotFoundError(f'Missing test file: {TEST_FILE.resolve()}')

def read_jsonl(path):
    rows = []
    with Path(path).open('r', encoding='utf-8') as f:
        for line in f:
            if line.strip():
                rows.append(json.loads(line))
    return rows

test_rows = read_jsonl(TEST_FILE)
print('Test rows:', len(test_rows))
print('Answer rows:', sum(bool(row.get('final_answer')) for row in test_rows))

tokenizer = AutoTokenizer.from_pretrained(ADAPTER_DIR, use_fast=True)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

base_model = AutoModelForCausalLM.from_pretrained(
    BASE_MODEL,
    torch_dtype=torch.float16,
    device_map='auto',
)
model = PeftModel.from_pretrained(base_model, ADAPTER_DIR)
model.eval()
print('Loaded adapter on device:', next(model.parameters()).device)


Test rows: 100
Answer rows: 100


Loading weights:   0%|          | 0/290 [00:00<?, ?it/s]

Loaded adapter on device: cuda:0


In [12]:
def generate_solution(problem, max_new_tokens=512):
    messages = [
        {"role": "system", "content": "You are a careful mathematical problem solver. Provide a complete solution with all necessary reasoning, and end with a line of the form 'Final Answer: <answer>'."},
        {"role": "user", "content": f"Solve the following problem:\n\n{problem}"},
    ]
    prompt = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
    inputs = tokenizer(prompt, return_tensors='pt', truncation=True, max_length=2048)
    inputs.pop('token_type_ids', None)
    device = next(model.parameters()).device
    inputs = {k: v.to(device) for k, v in inputs.items()}
    with torch.no_grad():
        output_ids = model.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            do_sample=False,
            pad_token_id=tokenizer.pad_token_id,
        )
    generated = output_ids[0, inputs['input_ids'].shape[-1]:]
    return tokenizer.decode(generated, skip_special_tokens=True).strip()


## Evaluate Base vs MATH-Only SFT

In [16]:
import re
import math
import numpy as np
import pandas as pd
from rouge_score import rouge_scorer
import nltk
from nltk.translate.bleu_score import sentence_bleu, SmoothingFunction
from tqdm.auto import tqdm
from contextlib import nullcontext

nltk.download('punkt', quiet=True)
nltk.download('punkt_tab', quiet=True)


def extract_last_boxed(text):
    marker = r'\boxed{'
    start = str(text or '').rfind(marker)
    if start == -1:
        return ''
    text = str(text)
    i = start + len(marker)
    depth = 1
    chars = []
    while i < len(text) and depth > 0:
        ch = text[i]
        if ch == '{':
            depth += 1
            chars.append(ch)
        elif ch == '}':
            depth -= 1
            if depth > 0:
                chars.append(ch)
        else:
            chars.append(ch)
        i += 1
    return ''.join(chars).strip() if depth == 0 else ''


def extract_answer(text):
    text = str(text or '').strip()
    matches = re.findall(r'Final Answer:\s*(.+)', text, flags=re.IGNORECASE)
    if matches:
        return matches[-1].strip()
    boxed = extract_last_boxed(text)
    if boxed:
        return boxed
    matches = re.findall(r'(?:answer is|answer:)\s*([^\n.]+)', text, flags=re.IGNORECASE)
    if matches:
        return matches[-1].strip()
    return ''


def normalize_answer(s):
    s = str(s or '').strip()
    s = extract_last_boxed(s) or s
    s = s.replace('\\,', '').replace(',', '')
    s = re.sub(r'\\(?:left|right)', '', s)
    s = re.sub(r'\s+', '', s)
    s = s.strip('$').strip().rstrip('.')
    return s.lower()


def get_numeric_value(s):
    s = normalize_answer(s)
    if not s:
        return None
    s = s.replace('\\$', '').replace('$', '').replace('\\%', '').replace('%', '').strip()
    frac = re.fullmatch(r'\\frac\{(-?\d+(?:\.\d+)?)\}\{(-?\d+(?:\.\d+)?)\}', s)
    if frac:
        denom = float(frac.group(2))
        return float(frac.group(1)) / denom if denom != 0 else None
    simple_frac = re.fullmatch(r'(-?\d+(?:\.\d+)?)/(-?\d+(?:\.\d+)?)', s)
    if simple_frac:
        denom = float(simple_frac.group(2))
        return float(simple_frac.group(1)) / denom if denom != 0 else None
    if re.fullmatch(r'-?\d+(?:\.\d+)?', s):
        return float(s)
    return None


def answers_match(pred_answer, gold_answer):
    pred_norm = normalize_answer(pred_answer)
    gold_norm = normalize_answer(gold_answer)
    if pred_norm and pred_norm == gold_norm:
        return True
    pred_val = get_numeric_value(pred_answer)
    gold_val = get_numeric_value(gold_answer)
    if pred_val is not None and gold_val is not None:
        return math.isclose(pred_val, gold_val, rel_tol=1e-9, abs_tol=1e-9)
    return False


def compute_metrics(rows, predictions):
    scorer = rouge_scorer.RougeScorer(['rougeL'], use_stemmer=True)
    smoothie = SmoothingFunction().method4
    answer_rows = 0
    correct = 0
    numeric_answer_rows = 0
    parsed_numeric_rows = 0
    percent_errors = []
    rouge_ls = []
    bleus = []
    length_ratios = []
    per_row = []

    for row, pred in zip(rows, predictions):
        ref = row.get('solution', '')
        gold = row.get('final_answer', '')
        pred_answer = extract_answer(pred)

        if ref:
            pred_words = len(pred.split())
            ref_words = len(ref.split())
            if ref_words:
                length_ratios.append(pred_words / ref_words)
            rouge_ls.append(scorer.score(ref, pred)['rougeL'].fmeasure)
            bleus.append(sentence_bleu([nltk.word_tokenize(ref)], nltk.word_tokenize(pred), smoothing_function=smoothie))

        is_correct = None
        pct_error = None
        gold_val = get_numeric_value(gold) if gold else None
        pred_val = get_numeric_value(pred_answer) if pred_answer else None
        if gold:
            answer_rows += 1
            is_correct = answers_match(pred_answer, gold)
            correct += int(is_correct)
            if gold_val is not None:
                numeric_answer_rows += 1
                if pred_val is not None:
                    parsed_numeric_rows += 1
                    if gold_val != 0:
                        pct_error = abs(gold_val - pred_val) / abs(gold_val) * 100
                    else:
                        pct_error = 0.0 if pred_val == 0 else np.inf
                    if np.isfinite(pct_error):
                        percent_errors.append(pct_error)

        per_row.append({
            'id': row.get('id'),
            'source': row.get('source'),
            'gold_answer': gold,
            'pred_answer': pred_answer,
            'gold_numeric': gold_val,
            'pred_numeric': pred_val,
            'correct': is_correct,
            'percent_error': pct_error,
            'prediction': pred,
        })

    return {
        'Answer Rows': answer_rows,
        'Accuracy': correct / answer_rows if answer_rows else 0.0,
        'Numeric Answer Rows': numeric_answer_rows,
        'Numeric Parse Coverage': parsed_numeric_rows / numeric_answer_rows if numeric_answer_rows else 0.0,
        'Average % Error (Parsed Numeric)': float(np.mean(percent_errors)) if percent_errors else np.nan,
        'Length Ratio': float(np.mean(length_ratios)) if length_ratios else 0.0,
        'ROUGE-L': float(np.mean(rouge_ls)) if rouge_ls else 0.0,
        'BLEU': float(np.mean(bleus)) if bleus else 0.0,
        '_per_row': per_row,
    }


def build_eval_rows(test_rows, total=20, seed=42):
    rows = [row for row in test_rows if row.get('final_answer')]
    rng = random.Random(seed)
    if len(rows) > total:
        rows = rng.sample(rows, k=total)
    rng.shuffle(rows)
    return rows


def score_model(label, rows, disable_lora=False, max_new_tokens=512):
    predictions = []
    context = model.disable_adapter() if disable_lora else nullcontext()
    print(f'Evaluating {len(rows)} samples on {label}...')
    with context:
        for row in tqdm(rows):
            predictions.append(generate_solution(row['problem'], max_new_tokens=max_new_tokens))
    metrics = compute_metrics(rows, predictions)
    details = metrics.pop('_per_row')
    return metrics, details


eval_rows = build_eval_rows(test_rows, total=20, seed=42)
print(f'Evaluating {len(eval_rows)} MATH final-answer test rows')

sft_metrics, sft_details = score_model('MATH-Only SFT Model (LoRA)', eval_rows, disable_lora=False)
base_metrics, base_details = score_model('Base Model (No LoRA)', eval_rows, disable_lora=True)

comparison_df = pd.DataFrame([
    {'Model': 'Base Model (No LoRA)', **base_metrics},
    {'Model': 'MATH-Only SFT Model (LoRA)', **sft_metrics},
])

print('\n=== COMPARISON TABLE ===')
display(comparison_df)

base_detail_df = pd.DataFrame(base_details)
sft_detail_df = pd.DataFrame(sft_details)

answer_compare_df = base_detail_df[
    ['id', 'gold_answer', 'pred_answer', 'gold_numeric', 'pred_numeric', 'correct', 'percent_error']
].rename(columns={
    'pred_answer': 'base_pred_answer',
    'pred_numeric': 'base_pred_numeric',
    'correct': 'base_correct',
    'percent_error': 'base_percent_error',
})
sft_answer_df = sft_detail_df[
    ['id', 'pred_answer', 'pred_numeric', 'correct', 'percent_error']
].rename(columns={
    'pred_answer': 'sft_pred_answer',
    'pred_numeric': 'sft_pred_numeric',
    'correct': 'sft_correct',
    'percent_error': 'sft_percent_error',
})
answer_compare_df = answer_compare_df.merge(sft_answer_df, on='id', how='left')
print('\n=== SCORED ANSWER ROWS ===')
display(answer_compare_df)


Evaluating 20 MATH final-answer test rows
Evaluating 20 samples on MATH-Only SFT Model (LoRA)...


  0%|          | 0/20 [00:00<?, ?it/s]

Evaluating 20 samples on Base Model (No LoRA)...


  0%|          | 0/20 [00:00<?, ?it/s]


=== COMPARISON TABLE ===


,Model,Answer Rows,Accuracy,Numeric Answer Rows,Numeric Parse Coverage,Average % Error (Parsed Numeric),Length Ratio,ROUGE-L,BLEU
0,Base Model (No LoRA),20,0.3,17,0.529412,153.262787,6.723098,0.242911,0.053377
1,MATH-Only SFT Model (LoRA),20,0.2,17,0.941176,183.271111,1.154544,0.400621,0.175133



=== SCORED ANSWER ROWS ===


,id,gold_answer,base_pred_answer,gold_numeric,base_pred_numeric,base_correct,base_percent_error,sft_pred_answer,sft_pred_numeric,sft_correct,sft_percent_error
0,math-787,400,\text{No valid solution},400.000000,NaN,False,NaN,\$625,625.000000,False,56.250000
1,math-67,-112,,-112.000000,NaN,False,NaN,38,38.000000,False,133.928571
2,math-1044,4,4,4.000000,4.0,True,0.000000,9,9.000000,False,125.000000
3,math-401,0,There is 1 point of intersection.,0.000000,NaN,False,NaN,1,1.000000,False,inf
4,math-800,\frac{35}{3},2.5,11.666667,2.5,False,78.571429,16,16.000000,False,37.142857
5,math-759,7,,7.000000,NaN,False,NaN,7,7.000000,True,0.000000
6,math-1677,-14,-10,-14.000000,-10.0,False,28.571429,-14,-14.000000,True,0.000000
7,math-1515,-9,,-9.000000,NaN,False,NaN,51,51.000000,False,666.666667
8,math-547,1,\text{Two},1.000000,NaN,False,NaN,\frac{-A}{B},NaN,False,NaN
9,math-1658,11,11,11.000000,11.0,True,0.000000,-12,-12.000000,False,209.090909


## Manual Side-By-Side Samples

In [ ]:
manual_rows = eval_rows[:3]
for row in manual_rows:
    print('=' * 120)
    print('ID:', row.get('id'))
    print('GOLD FINAL ANSWER:', row.get('final_answer'))
    print('\nPROBLEM:\n' + row['problem'])

    print('\n--- MATH-ONLY SFT MODEL OUTPUT ---')
    print(generate_solution(row['problem'], max_new_tokens=512))

    print('\n--- BASE MODEL OUTPUT ---')
    with model.disable_adapter():
        print(generate_solution(row['problem'], max_new_tokens=512))
    print()


ID: math-787
GOLD FINAL ANSWER: 400

PROBLEM:
Al, Betty, and Clare split $\$1000$ among them to be invested in different ways. Each begins with a different amount. At the end of one year they have a total of $\$1500$. Betty and Clare have both doubled their money, whereas Al has managed to lose $\$100$. What was Al's original portion?

--- MATH-ONLY SFT MODEL OUTPUT ---
Let $a$, $b$, and $c$ represent the amounts that Al, Betty, and Clare received respectively.

We know that $a+b+c=1000$, $ab+ac+bc=1500$, and $b=c-100$.

Substituting for $b$ into the second equation gives us $a(c-100)+c(c-100)=1500 \Rightarrow c^2-100c+a^2=1500$.

Since $a^2+b^2+c^2=3600$, we can subtract this from our last equation to get $(c^2-100c+a^2)-(c^2-100c+a^2)-2(a)(c)$ as well as $2(a-c)^2=2(1500-3600)=-4200$.

Thus, $a=\pm\sqrt{7} \cdot 30$, but since $a$ must be positive, we take $a=\boxed{\frac{30}{\sqrt{7}}}$.

Final Answer: \frac{30}{\sqrt{7}}

--- BASE MODEL OUTPUT ---
To solve this problem, we need to 

In [ ]:
with open('comparisons.txt', 'w', encoding='utf-8') as f:
    for idx, row in enumerate(eval_rows):
        f.write(f"Question {idx + 1} (ID: {row.get('id')})\n")
        f.write(f"Problem:\n{row['problem']}\n")
        f.write(f"Gold Answer: {row.get('final_answer')}\n")
        f.write("=" * 60 + "\n\n")

print('Successfully saved the 20 questions to comparisons.txt')

Successfully saved the 20 questions to comparisons.txt
